# 07 — Бэктест: модель против buy-and-hold

Практическая часть, шаг 2. Для каждого месяца из `D=21` торговых дней параметры
оцениваются по окну до точки решения, режим читается из границ прогона `RUN_ID`
при допуске `EPSILON`, в средней зоне применяется обученная стратегия ближайшего узла
того же прогона. Награда — реализованная лог-доходность `ln(P_τ/P₀)`.

Разделы:

1. **Синтетическая проверка**: смещение оценщиков и формула `σ`; согласие реализованной
   награды с теоретической стоимостью на синтетической истории.
2. **Реальные данные**: бэктест для базового окна, сравнение стратегий, интервалы.
3. Теория против практики (калибровка прогнозов).
4. Оптимизация окна с разделением in-sample / out-of-sample.
5. Чувствительность к `ε`, методу оценки `H` и комиссиям.

Методика — `docs/06-empirical.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
C1, C2, C3, C4, C5 = "#2a78d6", "#eb6834", "#1baf7a", "#8e63d2", "#b8860b"
INK, MUTED = "#0b0b0b", "#8a8a86"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titlesize": 11,
    "axes.grid": True, "grid.color": "#e8e8e4", "grid.linewidth": 0.8,
    "lines.linewidth": 1.6, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 9,
    "legend.frameon": False,
})
from tqdm.auto import tqdm

from osfbm.backtest import (STRATEGIES, by_period, calibration, decide_block, paired_difference,
                            regime_shares, run_backtest, summarize, window_search)
from osfbm.data import fetch_prices, month_blocks
from osfbm.estimation import ParamEstimate, estimate, simulate_log_prices
from osfbm.policy_bank import CASH, HOLD, POLICY, Boundaries, PolicyBank, ValueTable


## Параметры

In [ ]:
RUN_ID = "daily21_21_train20k_test20k_v1"
EPSILON = 0.06
CONFIDENCE = 0.95
TICKER = "^GSPC"
D = 21
WINDOW = 504
WINDOWS = (126, 252, 504, 756, 1008, 1260)
SPLIT_DATE = "2000-01-01"     # окно выбирается до этой даты, оценивается после
HURST_METHOD = "rs"
COST = 0.0                    # комиссия за сторону в лог-единицах, например 0.0005
EPSILONS = (0.04, 0.05, 0.06, 0.07)   # для раздела чувствительности
OUT = ROOT / "result_optimal_stopping" / "empirical" / RUN_ID / f"backtest_eps{EPSILON:g}"
OUT.mkdir(parents=True, exist_ok=True)
STRATEGY_LABELS = {"model": "модель (границы + стратегия)", "regimes_only": "только режимы",
                   "sign_drift": "знак сноса (H=1/2)", "buy_hold": "buy-and-hold", "oracle": "oracle"}
COLORS = dict(zip(STRATEGIES, (C1, C3, C4, C2, MUTED)))

## Прогон: границы, стратегии, стоимости

In [ ]:
RUN_DIR = ROOT / "result_optimal_stopping" / "runs" / RUN_ID
BOUNDARIES_CSV = RUN_DIR / "analysis" / f"epsilon_{EPSILON:g}" / f"confidence_{CONFIDENCE:g}" / "boundaries.csv"
if not BOUNDARIES_CSV.exists():
    raise SystemExit(f"Нет файла границ {BOUNDARIES_CSV}: выполните 04-grid-analysis.ipynb "
                     f"с RUN_ID={RUN_ID!r} и EPSILON={EPSILON:g}")
boundaries = Boundaries.from_csv(BOUNDARIES_CSV, EPSILON, RUN_ID)
print("Границы:", BOUNDARIES_CSV)
display(boundaries.frame().T.style.format(precision=2, na_rep="—"))

bank = PolicyBank(RUN_DIR)
values = ValueTable.from_run(RUN_DIR)
if bank.cfg.n_exercise != D:
    raise SystemExit(f"В прогоне {bank.cfg.n_exercise} дат решения, а D={D}")
coverage = bank.coverage()
print(f"Сохранённых стратегий: {coverage:.1%} узлов сетки H×μ = {len(bank.H_grid)}×{len(bank.mu_grid)}")
if coverage < 1:
    print("Прогон не завершён: ближайший узел может оказаться без стратегии.")
if bank.cfg.n_fine != D:
    print(f"Сетка наблюдений прогона {bank.cfg.n_fine} точек: дневной путь будет линейно "
          f"интерполирован на неё перед вычислением сигнатуры.")

## 1. Синтетическая проверка

### 1.1. Смещение оценщиков и формула σ

На симулированных окнах длины `WINDOW` с известными параметрами. Столбец
«σ по формуле СКО/D^H» показывает альтернативную формулу: она занижает σ в `D^{2H}` раз.

In [ ]:
rows = []
for H in (0.3, 0.5, 0.7):
    paths = simulate_log_prices(a=0.5, sigma=1.0, H=H, n_days=WINDOW, D=D, seed=100 + int(H * 100), n_paths=300)
    for method in ("rs", "dfa"):
        ests = [estimate(p, D, method) for p in paths]
        rows.append({"H истинное": H, "метод": method,
                     "Ĥ среднее": np.mean([e.H_raw for e in ests]), "Ĥ СКО": np.std([e.H_raw for e in ests]),
                     "σ̂ среднее (истина 1)": np.mean([e.sigma for e in ests]),
                     "σ по формуле СКО/D^H": np.mean([e.std_daily * D ** (-e.H) for e in ests]),
                     "â среднее (истина 0.5)": np.mean([e.a for e in ests]), "â СКО": np.std([e.a for e in ests])})
bias = pd.DataFrame(rows)
display(bias.style.format(precision=3))
bias.to_csv(OUT / "synthetic_estimator_bias.csv", index=False)

### 1.2. Реализованная награда против теории

Синтетическая история с известными `(a, σ, H)`. Два прогона: (а) стратегия при
**истинных** параметрах, где средняя награда должна совпасть с `σ·V̂(μ, H)` прогона
в пределах статистической ошибки; (б) полный пайплайн с **оценёнными** параметрами,
где потеря из-за оценивания видна явно.

In [ ]:
def synthetic_case(a, sigma, H, months=600, seed=1):
    logp_s = simulate_log_prices(a, sigma, H, months * D + WINDOW, D, seed=seed)[0]
    dates_s = pd.date_range("1950-01-01", periods=len(logp_s), freq="B")
    blocks = month_blocks(len(logp_s), D, warmup=WINDOW)
    truth = ParamEstimate(a=a, sigma=sigma, H=H, mu=a / sigma, H_raw=H, drift_daily=a / D,
                          std_daily=sigma * D ** (-H), n_returns=WINDOW, D=D, method="true")
    rewards = []
    for start, end in blocks:
        block = logp_s[start: end + 1]
        dec = decide_block(block, truth, bank, boundaries, values)
        rewards.append(0.0 if dec.tau_day == 0 else block[dec.tau_day] - block[0])
    rewards = np.asarray(rewards)
    res = run_backtest(logp_s, dates_s, window=WINDOW, D=D, bank=bank,
                       boundaries=boundaries, values=values, hurst_method=HURST_METHOD)
    return {"a": a, "σ": sigma, "H": H, "μ": a / sigma, "режим": boundaries.regime(H, a / sigma),
            "месяцев": len(blocks),
            "теория σ·V̂": sigma * values.value(H, a / sigma), "теория SE": sigma * values.se(H, a / sigma),
            "факт, истинные параметры": rewards.mean(), "SE": rewards.std(ddof=1) / np.sqrt(len(rewards)),
            "факт, оценённые параметры": res.r_model.mean(),
            "buy-and-hold": res.r_buy_hold.mean(), "теория B&H = a": a}

synthetic = pd.DataFrame([synthetic_case(0.0, 1.0, 0.7), synthetic_case(0.3, 1.0, 0.7),
                          synthetic_case(-0.5, 1.0, 0.3), synthetic_case(0.25, 1.0, 0.3),
                          synthetic_case(0.3, 1.0, 0.5), synthetic_case(-0.3, 1.0, 0.5)])
display(synthetic.style.format(precision=3))
synthetic.to_csv(OUT / "synthetic_theory_vs_practice.csv", index=False)

## 2. Реальные данные

### 2.1. Базовое окно

In [ ]:
prices = fetch_prices(TICKER)
logp = np.log(prices.to_numpy())
dates = prices.index
print(f"{TICKER}: {len(prices)} закрытий, {dates[0].date()} — {dates[-1].date()}")

with tqdm(total=len(month_blocks(len(logp), D, WINDOW)), desc=f"Бэктест W={WINDOW}") as bar:
    result = run_backtest(logp, dates, window=WINDOW, D=D, bank=bank, boundaries=boundaries,
                          values=values, cost=COST, hurst_method=HURST_METHOD,
                          progress=lambda i, n: bar.update(1))
result.to_csv(OUT / f"blocks_W{WINDOW}_{HURST_METHOD}.csv", index=False)
summary = summarize(result).rename(index=STRATEGY_LABELS)
print(f"Месяцев: {len(result)}, {result.decision_date.iloc[0].date()} — {result.end_date.iloc[-1].date()}")
display(summary.style.format(precision=4, na_rep="—"))
summary.to_csv(OUT / f"summary_W{WINDOW}.csv")
print("Доля месяцев по режимам:")
display(regime_shares(result).to_frame().style.format("{:.1%}"))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": (3, 1)})
for name in STRATEGIES:
    axes[0].plot(result.end_date, result[f"r_{name}"].cumsum(), color=COLORS[name],
                 label=STRATEGY_LABELS[name], linewidth=0.9 if name == "oracle" else 1.6)
axes[0].set(title=f"Накопленная лог-доходность, W={WINDOW}, ε={EPSILON:g}", ylabel="Σ ln(P_τ/P₀)")
axes[0].legend()
codes = result.regime.map({CASH: 0, POLICY: 1, HOLD: 2})
axes[1].scatter(result.decision_date, codes, c=codes, cmap="coolwarm", s=4)
axes[1].set(yticks=[0, 1, 2], yticklabels=["cash", "policy", "hold"], ylabel="режим", xlabel="дата")
fig.tight_layout(); fig.savefig(OUT / f"cumulative_W{WINDOW}.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
invested = result[result.tau_day > 0]
axes[0].hist(invested.tau_day, bins=np.arange(0.5, D + 1.5), color=C1)
axes[0].set(title="День продажи τ (купленные месяцы)", xlabel="день", ylabel="месяцев")
axes[1].scatter(result.mu, result.H, c=codes, cmap="coolwarm", s=8)
axes[1].set(title="Оценки (μ̂, Ĥ) по режимам", xlabel="μ̂", ylabel="Ĥ")
fig.tight_layout(); fig.savefig(OUT / f"tau_and_plane_W{WINDOW}.png", dpi=160, bbox_inches="tight")
plt.show()

### 2.2. Сравнение стратегий с интервалами

Парные разности месячных лог-доходностей: HAC-стандартная ошибка (Newey–West) и
стационарный бутстреп. Ниже — средние по десятилетиям.

In [ ]:
pairs = [("model", "buy_hold"), ("model", "regimes_only"), ("regimes_only", "buy_hold"),
         ("sign_drift", "buy_hold"), ("model", "sign_drift")]
differences = pd.DataFrame([paired_difference(result, a, b, seed=7) for a, b in pairs])
display(differences.style.format(precision=4))
differences.to_csv(OUT / f"paired_differences_W{WINDOW}.csv", index=False)
decades = by_period(result).rename(columns=STRATEGY_LABELS)
decades.index = decades.index.year
display(decades.style.format(precision=4))
decades.to_csv(OUT / f"by_decade_W{WINDOW}.csv")

## 3. Теория против практики

Прогноз модели на месяц: `0` в зоне cash, `â` в зоне hold, `σ̂·V̂(μ̂, Ĥ)` в средней
зоне; buy-and-hold прогнозирует `â`. Сравниваем средние и калибровку по децилям
прогноза.

In [ ]:
def mean_table(frame, label):
    return {"выборка": label, "месяцев": len(frame),
            "прогноз модели": frame.pred_model.mean(), "факт модели": frame.r_model.mean(),
            "SE факта": frame.r_model.std(ddof=1) / np.sqrt(len(frame)),
            "прогноз B&H (â)": frame.pred_buy_hold.mean(), "факт B&H": frame.r_buy_hold.mean()}

comparison = pd.DataFrame([mean_table(result, "все месяцы")] +
                          [mean_table(g, r) for r, g in result.groupby("regime")])
display(comparison.style.format(precision=4))
comparison.to_csv(OUT / f"theory_vs_practice_W{WINDOW}.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (pred, real, title, color) in zip(axes, (("pred_model", "r_model", "модель", C1),
                                              ("pred_buy_hold", "r_buy_hold", "buy-and-hold", C2))):
    cal = calibration(result, pred, real, bins=10)
    ax.errorbar(cal.pred_mean, cal.real_mean, yerr=1.96 * cal.real_se, fmt="o", color=color)
    lim = [min(cal.pred_mean.min(), cal.real_mean.min()), max(cal.pred_mean.max(), cal.real_mean.max())]
    ax.plot(lim, lim, color=MUTED, linewidth=0.8)
    ax.set(title=f"Калибровка: {title}", xlabel="прогноз (среднее в дециле)", ylabel="факт")
fig.tight_layout(); fig.savefig(OUT / f"calibration_W{WINDOW}.png", dpi=160, bbox_inches="tight")
plt.show()

## 4. Оптимизация окна

Окно выбирается по средней лог-награде модели **до** `SPLIT_DATE`; честная оценка
выбранного окна — столбцы `out_*`.

In [ ]:
search, per_window = window_search(logp, dates, windows=WINDOWS, split_date=SPLIT_DATE, D=D,
                                   bank=bank, boundaries=boundaries, values=values,
                                   cost=COST, hurst_method=HURST_METHOD)
chosen = search.attrs["chosen_window"]
print("Выбранное окно (in-sample):", chosen)
display(search.style.format(precision=4, na_rep="—"))
search.to_csv(OUT / "window_search.csv")

fig, ax = plt.subplots()
for part, marker in (("in", "o"), ("out", "s")):
    ax.plot(search.index, search[f"{part}_mean_model"], marker=marker, color=C1, label=f"модель, {part}")
    ax.plot(search.index, search[f"{part}_mean_buy_hold"], marker=marker, color=C2, linestyle="--",
            label=f"buy-and-hold, {part}")
ax.set(xscale="log", xlabel="окно W, дней", ylabel="средняя месячная лог-награда",
       title=f"Выбор окна: in-sample до {SPLIT_DATE}, out-of-sample после")
ax.legend()
fig.tight_layout(); fig.savefig(OUT / "window_search.png", dpi=160, bbox_inches="tight")
plt.show()
if chosen is not None:
    out_of_sample = per_window[chosen][per_window[chosen].decision_date >= pd.Timestamp(SPLIT_DATE)]
    print(f"Out-of-sample для W={chosen}: модель против buy-and-hold")
    display(pd.DataFrame([paired_difference(out_of_sample, "model", "buy_hold", seed=7)]).style.format(precision=4))

## 5. Чувствительность

Допуск `ε` меняет границы (читаются из того же прогона для каждого значения из
`EPSILONS`), метод оценки `H` меняет оценки, комиссии уменьшают награду каждого
купленного месяца. Окно — выбранное выше или базовое.

In [ ]:
W = chosen or WINDOW
rows = []
for eps in EPSILONS:
    csv = RUN_DIR / "analysis" / f"epsilon_{eps:g}" / f"confidence_{CONFIDENCE:g}" / "boundaries.csv"
    if not csv.exists():
        print(f"Нет границ для ε={eps:g}: пропуск (выполните 04 с этим допуском)")
        continue
    b = Boundaries.from_csv(csv, eps, RUN_ID)
    for method in ("rs", "dfa"):
        for cost in (0.0, 0.0005, 0.001):
            res = run_backtest(logp, dates, window=W, D=D, bank=bank, boundaries=b, values=values,
                               cost=cost, hurst_method=method)
            s = summarize(res)
            d = paired_difference(res, "model", "buy_hold", n_boot=500, seed=3)
            rows.append({"ε": eps, "метод H": method, "комиссия": cost, "W": W,
                         "модель, среднее": s.loc["model", "mean"], "B&H, среднее": s.loc["buy_hold", "mean"],
                         "разность": d["mean_diff"], "t (HAC)": d["t_hac"],
                         "ДИ низ": d["boot_low"], "ДИ верх": d["boot_high"],
                         "доля cash": (res.regime == CASH).mean(), "доля policy": (res.regime == POLICY).mean()})
sensitivity = pd.DataFrame(rows)
display(sensitivity.style.format(precision=4))
sensitivity.to_csv(OUT / "sensitivity.csv", index=False)
print("Результаты:", OUT)